# **Gold.Partner , User**

In [0]:

use catalog smart_odoo;

-- =====================================================
-- Gold Dim Partner
-- =====================================================

CREATE TABLE IF NOT EXISTS gold.dim_partner
(
    partner_id INT,
    company_id INT,
    parent_id INT,
    commercial_partner_id INT,
    country_id INT,
    state_id INT,
    industry_id INT,

    partner_name STRING,
    partner_type STRING,

    is_company BOOLEAN,
    active BOOLEAN,

    customer_rank INT,
    supplier_rank INT,

    email STRING,
    phone STRING,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP
)
USING DELTA;

-- =====================================================
-- MERGE INTO
-- =====================================================

MERGE INTO gold.dim_partner AS target

USING
(
    SELECT
        TRY_CAST(rp.partner_id AS INT) AS partner_id,
        TRY_CAST(rp.company_id AS INT) AS company_id,
        TRY_CAST(rp.parent_id AS INT) AS parent_id,
        TRY_CAST(rp.commercial_partner_id AS INT) AS commercial_partner_id,
        TRY_CAST(rp.country_id AS INT) AS country_id,
        TRY_CAST(rp.state_id AS INT) AS state_id,
        TRY_CAST(rp.industry_id AS INT) AS industry_id,

        rp.partner_name,

        CASE
            WHEN rp.customer_rank > 0 THEN 'Customer'
            WHEN rp.supplier_rank > 0 THEN 'Supplier'
            ELSE 'Other'
        END AS partner_type,

        rp.is_company,
        rp.active,

        TRY_CAST(rp.customer_rank AS INT) AS customer_rank,
        TRY_CAST(rp.supplier_rank AS INT) AS supplier_rank,

        rp.email,
        rp.phone,

        rp.created_at,
        rp.updated_at,

        current_timestamp() AS dwh_loaded_at

    FROM silver.res_partner rp

    WHERE rp.updated_at >=
    (
        SELECT COALESCE(
            MAX(updated_at) - INTERVAL 1 DAY,
            TIMESTAMP('1900-01-01')
        )
        FROM gold.dim_partner
    )

) AS source

ON target.partner_id = source.partner_id

WHEN MATCHED
AND target.updated_at < source.updated_at

THEN UPDATE SET
    target.partner_name = source.partner_name,
    target.company_id = source.company_id,
    target.parent_id = source.parent_id,
    target.commercial_partner_id = source.commercial_partner_id,
    target.country_id = source.country_id,
    target.state_id = source.state_id,
    target.industry_id = source.industry_id,
    target.partner_type = source.partner_type,
    target.is_company = source.is_company,
    target.active = source.active,
    target.customer_rank = source.customer_rank,
    target.supplier_rank = source.supplier_rank,
    target.email = source.email,
    target.phone = source.phone,
    target.created_at = source.created_at,
    target.updated_at = source.updated_at,
    target.dwh_loaded_at = current_timestamp()

WHEN NOT MATCHED THEN
INSERT
(
    partner_id,
    company_id,
    parent_id,
    commercial_partner_id,
    country_id,
    state_id,
    industry_id,
    partner_name,
    partner_type,
    is_company,
    active,
    customer_rank,
    supplier_rank,
    email,
    phone,
    created_at,
    updated_at,
    dwh_loaded_at
)

VALUES
(
    source.partner_id,
    source.company_id,
    source.parent_id,
    source.commercial_partner_id,
    source.country_id,
    source.state_id,
    source.industry_id,
    source.partner_name,
    source.partner_type,
    source.is_company,
    source.active,
    source.customer_rank,
    source.supplier_rank,
    source.email,
    source.phone,
    source.created_at,
    source.updated_at,
    current_timestamp()
);